In [1]:
import os

# Must be set before any huggingface_hub/datasets/sentence_transformers imports.
os.environ["HF_HUB_DISABLE_XET"] = "1"

from data.prepare import create_dataset_from_hf
from pipeline.chunk import chunk_documents, embed_chunks
from pipeline.vector_db import create_local_milvus_client
from pipeline.utils import get_available_prompts

In [2]:
embedding_model = "microsoft/harrier-oss-v1-0.6b"

In [4]:
docs = create_dataset_from_hf("galileo-ai/ragbench", "techqa", "validation")
print(f"Number of documents: {len(docs)}")
print("First document text:")
print(docs[0].text[:500])

Number of documents: 1520
First document text:
 RELEASE NOTES

ABSTRACT
 This document provides overview, upgrade, and system requirements for IBM Watson Explorer Version 10.0.0.4. 

CONTENT
 # 




 *  Overview 
 *  Upgrading 
 *  System Requirements 
 *  Known Issues 

OVERVIEW
IBM Watson Explorer Version 10.0.0.4 is a fix pack that addresses security issues. See the following Security Bulletins for more information: 

 * Security Bulletin: Vulnerability in OpenSSL affects Watson Explorer (CVE-2016-7055) [http://www.ibm.com/support/docview


In [5]:
bigger_chunks = chunk_documents(docs, chunk_size=512, chunk_overlap=20)
smaller_chunks = chunk_documents(docs, chunk_size=128, chunk_overlap=20)
print(f"Number of bigger chunks: {len(bigger_chunks)}")
print(f"Number of smaller chunks: {len(smaller_chunks)}")

Number of bigger chunks: 3968
Number of smaller chunks: 15896


In [6]:
print(bigger_chunks[0])
print("Chunk size:", len(bigger_chunks[0].get_content()))
print("-----------------------------")
print(smaller_chunks[0])
print("Chunk size:", len(smaller_chunks[0].get_content()))

Node ID: 0af68361-fec2-45b3-bf3e-69681dcbe9b8
Text: RELEASE NOTES  ABSTRACT  This document provides overview,
upgrade, and system requirements for IBM Watson Explorer Version
10.0.0.4.   CONTENT  #       *  Overview   *  Upgrading   *  System
Requirements   *  Known Issues   OVERVIEW IBM Watson Explorer Version
10.0.0.4 is a fix pack that addresses security issues. See the
following Security Bull...
Chunk size: 1437
-----------------------------
Node ID: 038d6173-463d-4e8a-926d-ba48cba909b6
Text: RELEASE NOTES  ABSTRACT  This document provides overview,
upgrade, and system requirements for IBM Watson Explorer Version
10.0.0.4.   CONTENT  #       *  Overview   *  Upgrading   *  System
Requirements   *  Known Issues   OVERVIEW IBM Watson Explorer Version
10.0.0.4 is a fix pack that addresses security issues. See the
following Security Bull...
Chunk size: 500


In [7]:
client = create_local_milvus_client(db_name="vector_db/vector_storage.db")

In [8]:
get_available_prompts(embedding_model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

{'query': '', 'document': '', 'web_search_query': 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ', 'sts_query': 'Instruct: Retrieve semantically similar text\nQuery: ', 'bitext_query': 'Instruct: Retrieve parallel sentences\nQuery: '}
['query', 'document', 'web_search_query', 'sts_query', 'bitext_query']
None


In [9]:
embedded_chunks = embed_chunks(bigger_chunks, embed_model=embedding_model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

100%|██████████| 3968/3968 [02:00<00:00, 33.00it/s]


In [10]:
embeddings: list = [chunk.embedding for chunk in embedded_chunks]
print(f"Embedding dimension: {len(embeddings[0])}")

Embedding dimension: 1024


In [13]:
print(type(embeddings[0]))

<class 'list'>


In [20]:
from pipeline.vector_db import create_milvus_collection
create_milvus_collection(client, collection_name="ragbench_techqa_validation", dimension=len(embeddings[0]))

Collection 'ragbench_techqa_validation' created with dimension 1024.


In [22]:
collection_name = "ragbench_techqa_validation"
collection_info = client.describe_collection(collection_name)

# Inspect schema
print(collection_info)

{'collection_name': 'ragbench_techqa_validation', 'auto_id': False, 'num_shards': 0, 'description': '', 'fields': [{'field_id': 100, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'is_primary': True}, {'field_id': 101, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'properties': {}, 'num_partitions': 0, 'enable_dynamic_field': True, 'enable_namespace': False}


I0000 00:00:1776252479.888397   59455 chttp2_transport.cc:1182] unix:/tmp/tmp8e0isp3a_vector_storage.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11, created_time:"2026-04-15T13:27:59.888381165+02:00"}
E0000 00:00:1776252479.888469   59455 chttp2_transport.cc:1210] unix:/tmp/tmp8e0isp3a_vector_storage.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
